In [9]:
df = spark.read.format("delta").load(
    "abfss://POC@onelake.dfs.fabric.microsoft.com/bronze_stream_lh.Lakehouse/Tables/dbo/raw"
)

display(df)

StatementMeta(, 92c388d4-14af-4030-bd20-e54486a0fcdd, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4b2506e6-f573-4f06-847f-bc1bbd86623e)

# Create Gate Transaction Silver

In [10]:
gate_df = df.filter(
    df.event_type == "gate_transaction"
)

StatementMeta(, 92c388d4-14af-4030-bd20-e54486a0fcdd, 12, Finished, Available, Finished, False)

In [11]:
from pyspark.sql.functions import col

gate_df = gate_df.select(
    "TransactionID",
    "GateID",
    "TruckID",
    "ContainerID",
    "EntryTime",
    "Direction",
    "CreatedUtc",
    "EventEnqueuedUtcTime"
)

StatementMeta(, 92c388d4-14af-4030-bd20-e54486a0fcdd, 13, Finished, Available, Finished, False)

In [12]:
from pyspark.sql.functions import to_timestamp

gate_df = gate_df \
    .withColumn(
        "EntryTime",
        to_timestamp("EntryTime")
    ) \
    .withColumn(
        "CreatedUtc",
        to_timestamp("CreatedUtc")
    )

StatementMeta(, 92c388d4-14af-4030-bd20-e54486a0fcdd, 14, Finished, Available, Finished, False)

In [13]:
dim_gate = spark.read.format("delta").load(
    "abfss://POC@onelake.dfs.fabric.microsoft.com/silver_lh.Lakehouse/Tables/dim_gate"
)

StatementMeta(, 92c388d4-14af-4030-bd20-e54486a0fcdd, 15, Finished, Available, Finished, False)

In [14]:
gate_df = gate_df.dropDuplicates(
    ["TransactionID"]
)

gate_df = gate_df.join(
    dim_gate,
    "GateID",
    "left"
)

StatementMeta(, 92c388d4-14af-4030-bd20-e54486a0fcdd, 16, Finished, Available, Finished, False)

In [15]:
gate_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("silver_gate_transaction")

StatementMeta(, 92c388d4-14af-4030-bd20-e54486a0fcdd, 17, Finished, Available, Finished, False)

# Create Crane Silver

In [16]:
crane_df = df.filter(
    df.event_type == "crane_event"
)

StatementMeta(, 92c388d4-14af-4030-bd20-e54486a0fcdd, 18, Finished, Available, Finished, False)

In [17]:
crane_df = crane_df.select(
    "CraneEventID",
    "CraneID",
    "VesselCallID",
    "EventTime",
    "MovesCompleted",
    "DowntimeMinutes",
    "CreatedUtc"
)

StatementMeta(, 92c388d4-14af-4030-bd20-e54486a0fcdd, 19, Finished, Available, Finished, False)

In [18]:
from pyspark.sql.functions import to_timestamp

crane_df = crane_df.withColumn(
    "EventTime",
    to_timestamp("EventTime")
)

StatementMeta(, 92c388d4-14af-4030-bd20-e54486a0fcdd, 20, Finished, Available, Finished, False)

In [19]:
crane_df = crane_df.dropDuplicates(
    ["CraneEventID"]
)

StatementMeta(, 92c388d4-14af-4030-bd20-e54486a0fcdd, 21, Finished, Available, Finished, False)

In [20]:
dim_crane = spark.read.format("delta").load(
    "abfss://POC@onelake.dfs.fabric.microsoft.com/silver_lh.Lakehouse/Tables/dim_crane"
)

StatementMeta(, 92c388d4-14af-4030-bd20-e54486a0fcdd, 22, Finished, Available, Finished, False)

In [21]:
crane_df = crane_df.join(
    dim_crane,
    "CraneID",
    "left"
)

StatementMeta(, 92c388d4-14af-4030-bd20-e54486a0fcdd, 23, Finished, Available, Finished, False)

In [22]:
crane_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("silver_crane_event")

StatementMeta(, 92c388d4-14af-4030-bd20-e54486a0fcdd, 24, Finished, Available, Finished, False)

# Create Yard Silver

In [23]:
yard_df = df.filter(
    df.event_type == "yard_snapshot"
)

StatementMeta(, 92c388d4-14af-4030-bd20-e54486a0fcdd, 25, Finished, Available, Finished, False)

In [24]:
yard_df = yard_df.select(
    "SnapshotID",
    "SnapshotTime",
    "YardBlockID",
    "OccupiedTEU",
    "CapacityTEU"
)

StatementMeta(, 92c388d4-14af-4030-bd20-e54486a0fcdd, 26, Finished, Available, Finished, False)

In [25]:
yard_df = yard_df.withColumn(
    "SnapshotTime",
    to_timestamp("SnapshotTime")
)

StatementMeta(, 92c388d4-14af-4030-bd20-e54486a0fcdd, 27, Finished, Available, Finished, False)

In [26]:
from pyspark.sql.functions import round

yard_df = yard_df.withColumn(
    "OccupancyPct",
    round(
        (col("OccupiedTEU") /
         col("CapacityTEU")) * 100,
        2
    )
)

StatementMeta(, 92c388d4-14af-4030-bd20-e54486a0fcdd, 28, Finished, Available, Finished, False)

In [27]:
dim_yard_block = spark.read.format("delta").load(
    "abfss://POC@onelake.dfs.fabric.microsoft.com/silver_lh.Lakehouse/Tables/dim_yard_block"
)

StatementMeta(, 92c388d4-14af-4030-bd20-e54486a0fcdd, 29, Finished, Available, Finished, False)

In [28]:
yard_df.join(
    dim_yard_block,
    "YardBlockID",
    "left"
)

StatementMeta(, 92c388d4-14af-4030-bd20-e54486a0fcdd, 30, Finished, Available, Finished, False)

DataFrame[YardBlockID: string, SnapshotID: string, SnapshotTime: timestamp, OccupiedTEU: string, CapacityTEU: string, OccupancyPct: double, Terminal: string, Zone: string, CapacityTEU: int]

In [29]:
yard_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("silver_yard_snapshot")

StatementMeta(, 92c388d4-14af-4030-bd20-e54486a0fcdd, 31, Finished, Available, Finished, False)

In [30]:
display(gate_df)

StatementMeta(, 92c388d4-14af-4030-bd20-e54486a0fcdd, 32, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, dbfe83bf-9616-4705-9cd0-c3626a92ab21)